# hPGC in vivo vs hPGCLC in vitro

*Ligand-receptor inference with LIANA+ to identify what the in vitro model is missing*.

Two settings cells need editing.

### Description

LIANA is run separately on each dataset with its own clusters. We then keep only
interactions where the germ cells are the receiver, pool all the senders, and
compare the two lists of ligand and receptor pairs. The somatic cell types in
the two datasets do not have to correspond, because the comparison is on pairs
and not on cell types.

### The six output categories

| class | meaning |
| --- | --- |
| `actionable` | in vivo only, hPGCLC has the receptor. Worth testing |
| `latent` | in vivo only, no receptor. Maturation problem, not a medium problem |
| `receptor_confounded` | receptor may be occupied by something you added |
| `supplied` | ligand is already in the medium as protein. Not a finding |
| `in_vitro_only` | the culture adds it and the embryo does not |
| `shared` | already working |

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, dpi_save=150, frameon=False)
print("liana", li.__version__)

## 1. Settings

`.X` must hold log-normalised expression in both objects, and both objects must
contain the full gene set rather than only highly variable genes. LIANA compares
expression levels directly, so scaled or batch-corrected values will give
nonsense.

In [ ]:
OUTDIR = "path/to/output/results_liana_GA_joao_v3" #change to your directory

INVIVO_H5AD  = "path_to_data/invivo_window_female.h5ad" #change to your directory
INVITRO_H5AD  = "path_to_data/invitro_integrated_scanpy_joao.h5ad" #change to your directory

INVIVO_CELLTYPE_KEY = "fine_label"     # obs column with the cluster labels
INVIVO_SAMPLE_KEY   = "donor_id"        # donor in vivo, line or batch in vitro
INVIVO_GERM  = "primordial germ cell"

INVITRO_CELLTYPE_KEY = "cell_type"
INVITRO_SAMPLE_KEY   = "sample"          # check this one
INVITRO_GERM = "PGCLC"

EXPR_PROP = 0.1     # a gene must be detected in this fraction of a cell type
MIN_CELLS = 10       # cell types smaller than this are dropped
N_PERMS   = 1000    # drop to 100 while you are still setting things up
SEED      = 42

RANK_CUTOFF       = 0.05   # flags the high confidence subset, does not decide presence
RECEPTOR_MIN_PROP = 0.1    # fraction of hPGCLCs that must express a receptor

MIN_SAMPLES = 2  # min samples a pair must be detected in to count as "in vivo"/"in vitro"


os.makedirs(OUTDIR, exist_ok=True)

# Every loop below iterates over this, so the right keys go with each object.
def datasets():
    return [("vivo",  vivo,  INVIVO_CELLTYPE_KEY,  INVIVO_SAMPLE_KEY,  INVIVO_GERM),
            ("vitro", vitro, INVITRO_CELLTYPE_KEY, INVITRO_SAMPLE_KEY, INVITRO_GERM)]

## 2. Medium Contents

**Edit based on the protocol used.**

**Ligands.** Recombinant proteins in the medium are never transcribed by any
cell, so a transcriptome comparison reports them as completely absent in vitro.
Without this list, they will come out near the top of your candidate
list.

**Receptors.** A saturating recombinant ligand occupies and downregulates
its own receptor, and that receptor is usually shared with other ligands
that really are missing. Example: recombinant EGF downregulates EGFR,
which is also the receptor for TGFA, HBEGF, AREG and EREG. Those are classified
as latent, meaning the cell cannot respond, when in fact the receptor is
present but occupied by what you added. The ligand list cannot catch this because
the mistake is on the receptor side.

**Other things to keep in mind.**

*Small molecules have no gene, so they cannot be filtered here at all.* Your
protocol can contain small molecules that activate pathways downstream, and a
missing ligand is not a real gap. Not encoded, just mentioned to keep in mind.
(Your protocol contains CHIR99021, so a missing WNT ligand is not a real gap.)

*Serum contain undefined growth factors.* If in your medium, treat the whole
candidate list as approximate, because you cannot enumerate what is actually
in there.

In [ ]:
# ligands supplied as recombinant protein
MEDIUM_FACTORS = {
    "BMP2":   "substitute for BMP4?",   
    "INHBA":  "Activin A",
    "KITLG":  "SCF",
    "EGF":    "EGF",
    "INS":    "Insulin (B27)",
    "TF":     "Transferrin (B27)",
    # CHECK AND ADD OTHERS
}

# receptors those factors occupy, so a low reading may be your protocol
MEDIUM_RECEPTORS = {
    "BMPR1A": "occupied by recombinant BMP4",
    "BMPR1B": "occupied by recombinant BMP4",
    "BMPR2":  "occupied by recombinant BMP4",
    "ACVR2A": "shared BMP and activin type II receptor",
    "ACVR2B": "shared BMP and activin type II receptor",
    "ACVR1B": "occupied by Activin A",
    "TGFBR1": "shared type I receptor",
    "KIT":    "occupied by recombinant SCF",
    "EGFR":   "occupied by recombinant EGF",
    "INSR":   "occupied by insulin (B27)",
    "IGF1R":  "insulin (B27) can cross-react at supraphysiological doses",
    "TFRC":   "occupied by transferrin (B27)",
    # CHECK AND ADD OTHERS
}

# --- non-protein medium components (B27 hormones/vitamins) ---
# These act through intracellular nuclear receptors, not surface receptor
# complexes formed with a secreted protein ligand - so they have no
# ligand_complex/receptor_complex representation in a standard
# protein-protein LR resource and will never appear in a.uns['liana_res'].
# Adding them to MEDIUM_FACTORS/MEDIUM_RECEPTORS would silently match
# nothing. Kept here only as a record of what else may be acting on the
# germ cells
NON_PROTEIN_MEDIUM_COMPONENTS = {
    "Corticosterone":        "NR3C1 (glucocorticoid receptor); some affinity for NR3C2",
    "Progesterone":          "PGR (progesterone receptor)",
    "Triiodothyronine (T3)": "THRA / THRB (thyroid hormone receptors)",
    "Retinyl acetate":       "converted to retinoic acid -> RARA/RARB/RARG (+ RXR heterodimer partners)",
}

This next block must run AFTER liana. It checks whether each ligand/receptor related to the medium factors is actually present in the results of each dataset.

In [ ]:
for name, a, ct_key, s_key, germ in datasets():
    lig_present = set(a.uns["liana_res"]["ligand_complex"].astype(str).str.split("_").sum())
    rec_present = set(a.uns["liana_res"]["receptor_complex"].astype(str).str.split("_").sum())
    print(f"{name}: KITLG={'KITLG' in lig_present}, EGF={'EGF' in lig_present}, "
          f"INS={'INS' in lig_present}, TF={'TF' in lig_present}, "
          f"KIT={'KIT' in rec_present}, EGFR={'EGFR' in rec_present}, "
          f"INSR={'INSR' in rec_present}, TFRC={'TFRC' in rec_present}")

## 3. Load

If `.X max` comes out above about 50 the matrix is raw counts, and you need to
normalise before going further.

In [ ]:
vivo  = sc.read_h5ad(INVIVO_H5AD)
vitro = sc.read_h5ad(INVITRO_H5AD)

for a, ct_key, s_key in [(vivo,  INVIVO_CELLTYPE_KEY,  INVIVO_SAMPLE_KEY),
                         (vitro, INVITRO_CELLTYPE_KEY, INVITRO_SAMPLE_KEY)]:
    a.var_names_make_unique()
    a.obs[ct_key] = a.obs[ct_key].astype(str)
    a.obs[s_key]  = a.obs[s_key].astype(str)

for name, a, ct_key, s_key, germ in datasets():
    xmax = a.X.data.max() if hasattr(a.X, "data") else a.X.max()
    print(f"{name:6s} {a.n_obs:6d} cells x {a.n_vars} genes | .X max {xmax:.2f} | "
          f"{int((a.obs[ct_key] == germ).sum())} germ cells "
          f"| {a.obs[s_key].nunique()} samples")

for name, a, ct_key, s_key, germ in datasets():
    print(f"\n--- {name}: {ct_key} by {s_key} ---")
    print(a.obs.groupby([ct_key, s_key], observed=True)
          .size().unstack(fill_value=0).to_string())

### Check stages and sexes

Both change how the result should be read.

**Stage.** If the object covers 6 to 9 pcw rather than a single timepoint, the
DAZL and DDX4 split in the next section is important.

**Sex.** Human gonads at this stage are in the middle of sex determination. If
the donors are mixed, the variance between them is sex rather than donor noise.
The germ cells are still similar here but the somatic senders are not.

In [ ]:
for col in vivo.obs.columns:
    if col.lower() in ("week", "week_pf", "pcw", "day_pf", "stage",
                       "development_stage", "age", "timepoint", "sex"):
        print(f"--- {col} ---")
        print(pd.crosstab(vivo.obs[INVIVO_SAMPLE_KEY], vivo.obs[col]).to_string())
        print()

## 4. Stage matching

Day 4 hPGCLCs is similar to the very start of germ cell development. hPGCs at
6 to 9 pcw are further along. If compared directly much of what comes out as
missing in vitro is a signal the cultured cells are too young to receive,
which no medium change will fix.

DAZL and DDX4 switch on relatively late, so they split the in vivo germ cells
into an earlier and a later group. We compare against the earlier one.

Look at the counts this prints. If `PGC_early` is below about 30 cells there is
no early population to compare against, so set `MATCH_STAGE = False` and keep in
mind that stage will affect part of your result.

In [ ]:
MATCH_STAGE = False

print("proportion of germ cells expressing each marker")
for g in ["DAZL", "DDX4", "PIWIL2", "DPPA3", "NANOS3", "SOX17"]:
    line = f"  {g:<8}"
    for name, a, ct_key, s_key, germ in datasets():
        if g in a.var_names:
            v = a[a.obs[ct_key] == germ, g].X
            v = v.toarray().ravel() if hasattr(v, "toarray") else np.asarray(v).ravel()
            line += f"   {name} {(v > 0).mean():6.1%}"
        else:
            line += f"   {name}    n/a"
    print(line)

if MATCH_STAGE:
    late = np.zeros(vivo.n_obs, dtype=bool)
    for g in ["DAZL", "DDX4"]:
        v = vivo[:, g].X
        v = v.toarray().ravel() if hasattr(v, "toarray") else np.asarray(v).ravel()
        late |= (v > 0)

    germ_mask = (vivo.obs[INVIVO_CELLTYPE_KEY] == INVIVO_GERM).values
    lab = vivo.obs[INVIVO_CELLTYPE_KEY].copy()
    lab[germ_mask & ~late] = "PGC_early"
    lab[germ_mask &  late] = "PGC_late"
    vivo.obs[INVIVO_CELLTYPE_KEY] = lab.values
    INVIVO_GERM = "PGC_early"

    print(f"\nPGC_early {int((lab == 'PGC_early').sum())}   "
          f"PGC_late {int((lab == 'PGC_late').sum())}")
    print("comparing against PGC_early")
    print("per donor:", vivo.obs[vivo.obs[INVIVO_CELLTYPE_KEY] == "PGC_early"]
          [INVIVO_SAMPLE_KEY].value_counts().to_dict())

**Notes.** PGC_early is 36 and late 277. Should we still set it to False, and keep them? 

## 5. Run LIANA

It runs twice per dataset. The first run pools all cells and gives the scores.
The second splits by sample so we can see how many donors support each interaction,
which matters because pooling cells across donors treats cells as independent
when they are not.

This is slow. Set `N_PERMS = 100` for testing the code.

In [ ]:
N_PERMS = 1000

for name, a, ct_key, s_key, germ in datasets():
    print(f"--- {name}, pooled ---")
    li.mt.rank_aggregate(
        a, groupby=ct_key, expr_prop=EXPR_PROP, min_cells=MIN_CELLS,
        n_perms=N_PERMS, seed=SEED, use_raw=False, verbose=True,
    )
    print(f"{a.uns['liana_res'].shape[0]} interactions\n")

    print(f"--- {name}, per sample ---")
    li.mt.rank_aggregate.by_sample(
        a, sample_key=s_key, groupby=ct_key, expr_prop=EXPR_PROP,
        min_cells=MIN_CELLS, n_perms=N_PERMS, seed=SEED, use_raw=False,
        key_added="liana_by_sample", verbose=False,
    )
    print(f"{a.uns['liana_by_sample'].shape[0]} rows\n")

In [ ]:
vivo.uns["liana_res"].sort_values("magnitude_rank").head()

**Rank columns.**

`magnitude_rank` says how strongly the pair is expressed. `specificity_rank` says
whether the pair is particular to these two clusters or is happening everywhere.
**Both are ranks, so smaller is stronger.**

Magnitude rewards abundance, so sorting by it alone moves housekeeping genes to
the top. Neither rank decides whether an interaction is present. Presence is
decided by `EXPR_PROP`, which is an absolute threshold and therefore comparable
between the two datasets.

## 6. One row per ligand and receptor pair

Keep interactions where the germ cells are the receiver, then pool the senders.
Pooling is what lets the two datasets have completely different somatic cell
types, and it affects nothing here because the intervention is a medium component.
It does not matter which cell would have made the ligand in the embryo.

The replication denominator counts only samples where the germ population cleared
`MIN_CELLS`. A donor with too few germ cells can never support any interaction,
so counting it would make `all_samples` unreachable.

In [ ]:
def match_all(name, lookup):
    """Return every lookup value matching a subunit of a (possibly complex) ligand/receptor
    name, or None if there are no matches. Avoids silently dropping subunits in complexes."""
    hits = [lookup[p] for p in str(name).split("_") if p in lookup]
    return hits if hits else None

lr = {}
for name, a, ct_key, s_key, germ in datasets():
    res = a.uns["liana_res"]
    incoming = res[res["target"] == germ].copy()
    incoming["high_conf"] = incoming["magnitude_rank"] <= RANK_CUTOFF
    g = (incoming.groupby(["ligand_complex", "receptor_complex"], observed=True)
         .agg(best_rank=("magnitude_rank", "min"),
              high_conf=("high_conf", "any"),
              senders=("source", lambda s: ", ".join(sorted(set(s)))),
              n_senders=("source", "nunique")))
    # how many samples support each pair
    per_sample = a.uns["liana_by_sample"]
    per_sample = per_sample[per_sample["target"] == germ]
    n_sup = (per_sample.groupby(["ligand_complex", "receptor_complex"], observed=True)
             [s_key].nunique())
    germ_counts = a.obs[a.obs[ct_key] == germ][s_key].value_counts()
    n_testable = int((germ_counts >= MIN_CELLS).sum())
    g["n_samples"] = n_sup.reindex(g.index).fillna(0).astype(int)
    g["all_samples"] = g["n_samples"] == n_testable

    # flatten categorical MultiIndex to plain strings — vivo/vitro frames can have
    # different category sets (different resources/filtering), which makes the
    # later outer join unreliable if left categorical
    g.index = pd.MultiIndex.from_tuples(
        [(str(l), str(r)) for l, r in g.index], names=g.index.names
    )

    lr[name] = g
    print(f"{name:6s} {len(g):5d} incoming pairs | {n_testable} testable samples | "
          f"{int(g['all_samples'].sum())} pairs in all of them")
print(f"\nshared: {len(lr['vivo'].index.intersection(lr['vitro'].index))}")

## 7. Can the hPGCLCs receive it?

This filter separates a candidate worth testing from a dead end.
Adding a factor to the medium does nothing if the cells do not carry the
receptor.

For receptors made of several subunits we require **all** of them in the **same
cell**, which is how LIANA treats complexes. A complex with a subunit missing
from the object cannot be scored and is treated as unavailable.

In [ ]:
germ_vitro = vitro[vitro.obs[INVITRO_CELLTYPE_KEY] == INVITRO_GERM]
rec_prop = {}
for r in sorted(set(lr["vivo"].index.get_level_values("receptor_complex"))):
    parts = str(r).split("_")
    if not all(p in vitro.var_names for p in parts):
        rec_prop[r] = np.nan
        continue
    m = germ_vitro[:, parts].X
    m = m.toarray() if hasattr(m, "toarray") else np.asarray(m)
    rec_prop[r] = float((m > 0).all(axis=1).mean())
rec_prop = pd.Series(rec_prop)
print(f"{len(rec_prop)} receptors checked on hPGCLCs, "
      f"{int((rec_prop >= RECEPTOR_MIN_PROP).sum())} expressed above "
      f"{RECEPTOR_MIN_PROP:.0%}, {int(rec_prop.isna().sum())} not scorable")

## 8. Compare and Classify

Every ligand–receptor pair is tested against these conditions **in order** — the pair gets labelled by the **first** one that's true.

| # | Condition | Class | Meaning |
|---|---|---|---|
| 1 | Confidently seen in both vivo **and** vitro | `shared` | Already working in vitro — nothing to fix here. |
| 2 | Confidently seen in vitro **only** | `in vitro only` | Happening in your dish but wasn't a confirmed in vivo signal — not your target for medium changes. |
| 3 | Confidently seen in vivo, **and** ligand already in medium | `supplied in medium` | Already covered — no action needed. |
| 4 | Confidently seen in vivo, **and** receptor expressed on hPGCLCs | `actionable` | Real signal in vivo, cells can respond to it, ligand is missing from the dish. **This is your priority list.** |
| 5 | Confidently seen in vivo, **and** receptor overlaps with something in the medium | `receptor confounded by medium` | Same idea as actionable, but adding the ligand may be hard to interpret since something already engages that receptor. (`flag_receptor_confounded` still flags this overlap on `actionable` rows too.) |
| 6 | Confidently seen in vivo, but receptor **couldn't be checked** | `not scorable` | Worth a manual look. |
| 7 | Confidently seen in vivo, but receptor **confirmed absent** | `latent` | Real in vivo signal, but hPGCLCs aren't currently equipped to respond even if you added the ligand. |
| 8 | None of the above | `undetected` | Too weak / unreproducible either way — excluded from the gap analysis. |


In [ ]:
comp = lr["vivo"].join(lr["vitro"], how="outer", lsuffix="_vivo", rsuffix="_vitro")
comp = comp.reset_index()

# --- confidence + reproducibility gate "in vivo" / "in vitro" ---
comp["detected_vivo"]  = comp["best_rank_vivo"].notna()   # loose: passed filtering at all
comp["detected_vitro"] = comp["best_rank_vitro"].notna()
comp["in_vivo"]  = comp["high_conf_vivo"].fillna(False)  & (comp["n_samples_vivo"].fillna(0)  >= MIN_SAMPLES)
comp["in_vitro"] = comp["high_conf_vitro"].fillna(False) & (comp["n_samples_vitro"].fillna(0) >= MIN_SAMPLES)

comp["receptor_prop_pgclc"] = comp["receptor_complex"].map(rec_prop)
comp["receptor_scorable"]   = comp["receptor_prop_pgclc"].notna()
comp["receptor_available"]  = comp["receptor_prop_pgclc"] >= RECEPTOR_MIN_PROP  # NaN-safe -> False

# --- medium matches: keep every matching subunit of a complex, not just the first ---
comp["medium_supplied_hits"] = comp["ligand_complex"].apply(lambda l: match_all(l, MEDIUM_FACTORS))
comp["medium_supplied"] = comp["medium_supplied_hits"].apply(lambda h: h[0] if h else None)
comp["medium_supplied_partial"] = comp.apply(
    lambda row: row["medium_supplied_hits"] is not None
                and len(row["medium_supplied_hits"]) < len(str(row["ligand_complex"]).split("_")),
    axis=1)

comp["receptor_confounded_hits"] = comp["receptor_complex"].apply(lambda r: match_all(r, MEDIUM_RECEPTORS))
comp["receptor_confounded"] = comp["receptor_confounded_hits"].apply(lambda h: h[0] if h else None)

# --- independent flags, so overlaps (e.g. actionable + confounded) aren't hidden ---
comp["flag_medium_supplied"]     = comp["medium_supplied"].notna()
comp["flag_receptor_available"]  = comp["receptor_available"]
comp["flag_receptor_confounded"] = comp["receptor_confounded"].notna()

comp["class"] = np.select(
    [comp["in_vivo"] & comp["in_vitro"],
     comp["in_vitro"] & ~comp["in_vivo"],
     comp["in_vivo"] & comp["flag_medium_supplied"],
     comp["in_vivo"] & comp["flag_receptor_available"],
     comp["in_vivo"] & comp["flag_receptor_confounded"],
     comp["in_vivo"] & ~comp["receptor_scorable"],
     comp["in_vivo"]],
    ["shared", "in vitro only", "supplied in medium", "actionable",
     "receptor confounded by medium", "not scorable", "latent"],
    default="undetected")

print(comp["class"].value_counts())
print(f"\nreceptor-confounded overlapping with actionable: "
      f"{int((comp['flag_receptor_confounded'] & comp['flag_receptor_available']).sum())}")
print(f"partial medium matches (complex only partly supplied): "
      f"{int(comp['medium_supplied_partial'].sum())}")

In [ ]:
# A: how many receptor partners does each ligand actually have?
pairs_per_ligand = comp.groupby("ligand_complex").size()
print(pairs_per_ligand.describe())
print((pairs_per_ligand == 1).sum(), "out of", len(pairs_per_ligand), "ligands have only 1 receptor pair")

# B: does in_vitro detection status ever vary within one ligand's own rows?
print(comp.groupby("ligand_complex")["in_vitro"].nunique().value_counts())

### How much does the receptor threshold decide?

`RECEPTOR_MIN_PROP` is the one parameter that moves a pair between actionable and
latent, so it is important. Anything that is only actionable at a lenient
setting is not a result.

The thresholds are nested, so the pairs that survive every setting are just the
ones that survive the strictest.

In [ ]:
pool = comp["in_vivo"] & ~comp["in_vitro"] & comp["medium_supplied"].isna()

for cut in (0.05, 0.10, 0.15, 0.20, 0.30):
    print(f"  receptor cutoff {cut:.2f}  ->  "
          f"{int((pool & (comp['receptor_prop_pgclc'] >= cut)).sum()):4d} actionable")

comp["robust"] = pool & (comp["receptor_prop_pgclc"] >= 0.30)
print(f"\nactionable at every cutoff: {int(comp['robust'].sum())}")

## 9. Results

The candidate list. Sorted so that pairs found in every donor come first, then by
how well the hPGCLCs express the receptor, then by in vivo strength.

In [ ]:
gaps = (comp[comp["class"] == "actionable"]
        .sort_values(["all_samples_vivo", "receptor_prop_pgclc", "best_rank_vivo"],
                     ascending=[False, False, True]))

gaps[["ligand_complex", "receptor_complex", "senders_vivo", "best_rank_vivo",
      "high_conf_vivo", "n_samples_vivo", "all_samples_vivo",
      "receptor_prop_pgclc", "robust"]].head(30)

Signals the culture applies that the embryo does not. Removing something can
matter as much as adding something.

In [ ]:
artefacts = (comp[comp["class"] == "in vitro only"]
             .sort_values(["all_samples_vitro", "best_rank_vitro"],
                          ascending=[False, True]))
artefacts[["ligand_complex", "receptor_complex", "senders_vitro",
           "best_rank_vitro", "n_samples_vitro"]].head(20)

Receptors that may only look absent because of the medium. These need a protocol
test, meaning omit the factor and restain, rather than a reagent order.

In [ ]:
comp.loc[comp["class"] == "receptor confounded by medium",
         ["ligand_complex", "receptor_complex", "senders_vivo",
          "receptor_prop_pgclc", "receptor_confounded"]].head(20)

And what the medium already covers, kept visible so it is not silently dropped.

In [ ]:
comp.loc[comp["class"] == "supplied in medium",
         ["ligand_complex", "receptor_complex", "senders_vivo", "medium_supplied"]]

## 10. Plots

Which in vivo ligands are recapitulated in vitro and which are not, counted over
their receptor pairs.

For each ligand: how many distinct receptor groups it engages in vivo that are
missing in vitro and whose receptor the hPGCLCs still carry (n_gaps), out of
everything it does in vivo (n_invivo). The fraction distinguishes a ligand
missing 1 of 4 groups from one missing 4 of 4.

Every class representing a pair seen in vivo. "in vitro only" is excluded:
those have no in vivo counterpart to be missing from. These strings are the
np.select labels from section 8. Make sure to change actionable if you used
something else when you fixed my bug. Added a check anyway.

In [ ]:
import re

VIVO_CLASSES = ["actionable", "latent", "supplied in medium",
                "receptor confounded by medium", "not scorable", "shared"]
assert (comp["class"] == "actionable").any(), \
    f"no rows classed 'actionable'; present labels: {sorted(comp['class'].unique())}"

COLLAPSE_COMPLEXES = True

def _family(name):
    out = set()
    for part in str(name).split("_"):
        stem = re.sub(r"\d+[A-Z]?$", "", part)
        out.add(stem if len(stem) >= 3 else part)
    return "+".join(sorted(out))

invivo = comp[comp["class"].isin(VIVO_CLASSES)].copy()
invivo["receptor_group"] = (invivo["receptor_complex"].astype(str).map(_family)
                            if COLLAPSE_COMPLEXES
                            else invivo["receptor_complex"].astype(str))

n_gaps = (invivo[invivo["class"] == "actionable"]
          .drop_duplicates(["ligand_complex", "receptor_group"])
          .groupby("ligand_complex").size().sort_values(ascending=False))

n_invivo = (invivo.drop_duplicates(["ligand_complex", "receptor_group"])
            .groupby("ligand_complex").size().reindex(n_gaps.index))

summary = pd.DataFrame({"n_gaps": n_gaps, "n_invivo": n_invivo})
summary["frac_missing"] = summary["n_gaps"] / summary["n_invivo"]

n_partial = int((summary["frac_missing"] < 1.0).sum())
print(f"{len(summary)} ligands have at least one testable gap, "
      f"{int(summary.n_gaps.sum())} groups in total. "
      f"{len(summary) - n_partial} are missing every group they use in vivo, "
      f"{n_partial} are missing only some, so the fraction is informative there.")

summary

In [ ]:
# Bars are the raw count, not the fraction: with most ligands missing everything
# they use, the fraction is flat across the top of the ranking. Read frac_missing
# from the summary table above instead.
TOP_N = 20
DATASET_TAG = "v4"
top = summary.head(TOP_N)[::-1]  # top N by n_gaps (summary is already sorted that way), reversed for barh

fig, axis = plt.subplots(figsize=(6, max(3, 0.3 * len(top))))
axis.barh(range(len(top)), top["n_gaps"], color="#b2182b")
axis.set_yticks(range(len(top)))
axis.set_yticklabels(top.index)
axis.set_xlabel("receptor groups present in vivo, missing in vitro")
axis.set_ylabel("")

max_gap = top["n_gaps"].max()
axis.set_xlim(0, max_gap * 1.15)  # headroom so labels don't get clipped at the right edge
for i, r in enumerate(top.itertuples()):
    axis.text(r.n_gaps + max_gap * 0.02, i, f"{r.n_gaps} of {r.n_invivo}",
              va="center", fontsize=8, color="#555555")

plt.savefig(os.path.join(OUTDIR, f"ligand_gaps_{DATASET_TAG}.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# the rows where the fraction carries information
summary[summary["frac_missing"] < 1.0].sort_values("frac_missing")

In [ ]:
# Only the ligands scoring below 1.0 are plotted. The rest are tied at 1.0, so a
# bar for them carries no information and would crowd out the ones that do.
# Bar length is the fraction, the label on the right gives the underlying counts,
# because 1 of 2 and 3 of 6 are the same fraction from very different evidence.
fracs = (summary[summary["frac_missing"] < 1.0]
         .sort_values("frac_missing", ascending=False))

if len(fracs) == 0:
    print("nothing to plot: every ligand with a gap is missing all of its "
          "receptor groups, so frac_missing is 1.0 throughout.")
else:
    print(f"{len(fracs)} of {len(summary)} ligands are only partly missing, "
          f"the other {len(summary) - len(fracs)} are at 1.0 and not shown.")

    fig, axis = plt.subplots(figsize=(6.5, max(3, 0.32 * len(fracs))))
    axis.barh(range(len(fracs)), fracs["frac_missing"], color="#b2182b", height=0.72)
    axis.set_yticks(range(len(fracs)))
    axis.set_yticklabels(fracs.index)
    axis.invert_yaxis()
    axis.set_xlim(0, 1)
    axis.set_xlabel("fraction of this ligand's in vivo receptor groups missing in vitro")
    for i, r in enumerate(fracs.itertuples()):
        axis.text(r.frac_missing + 0.015, i, f"{r.n_gaps} of {r.n_invivo}",
                  va="center", fontsize=8, color="#555555")
    plt.savefig(os.path.join(OUTDIR, f"ligand_frac_missing_{DATASET_TAG}.png"),
                dpi=150, bbox_inches="tight")
    plt.show()

fracs

In [ ]:
summary.to_csv(os.path.join(OUTDIR, f"summary_{DATASET_TAG}.csv"), index=False)
summary

## L-R Dotplot

The bar chart shows one bar per ligand, but a dotplot shows one row per ligand–receptor pair, with sender cell type as columns. It shows us which sender drives each interaction, and how confident/specific it is.

In [ ]:
import liana as li
import plotnine as p9

TOP_N = 20
DATASET_TAG = "v4"

top_ligands = summary["n_gaps"].sort_values(ascending=False).head(TOP_N).index.tolist()

dot_pairs = (
    comp[(comp["class"] == "actionable") & comp["ligand_complex"].isin(top_ligands)]
    .sort_values("best_rank_vivo")
    .drop_duplicates("ligand_complex")
    .set_index("ligand_complex")
    .loc[top_ligands]
    .reset_index()
)
print(f"{len(dot_pairs)} ligand-receptor pairs selected for the dotplot "
      f"(1 per ligand, top {TOP_N} by gap count)")

_, vivo_adata, vivo_ct_key, vivo_s_key, GERM_VIVO = next(
    d for d in datasets() if d[0] == "vivo"
)

allowed_pairs = set(zip(dot_pairs["ligand_complex"], dot_pairs["receptor_complex"]))

def _exact_pair_filter(row):
    """Called once per row of liana_res (via df.apply(..., axis=1)) - must
    return a single True/False for that row, not a Series."""
    return (row["ligand_complex"], row["receptor_complex"]) in allowed_pairs

fig = li.pl.dotplot(
    adata=vivo_adata,
    uns_key="liana_res",
    colour="magnitude_rank",
    size="specificity_rank",
    inverse_colour=True,
    inverse_size=True,
    target_labels=[GERM_VIVO],
    ligand_complex=dot_pairs["ligand_complex"].tolist(),
    receptor_complex=dot_pairs["receptor_complex"].tolist(),
    filter_fun=_exact_pair_filter,
    cmap="magma",
    size_range=(2, 9),
    figure_size=(8, max(4, 0.3 * len(dot_pairs))),
)

fig = (
    fig
    + p9.theme(
        strip_text=p9.element_text(size=8, angle=90),     # top: source labels, smaller + vertical
        axis_text_x=p9.element_blank(),                    # bottom: drop the repeated per-panel target text
        axis_ticks_x=p9.element_blank(),                   # and the tick marks that went with it
        axis_title_x=p9.element_text(size=11, angle=0),    # keep the axis title itself horizontal
        figure_size=(11, max(7, 0.3 * len(dot_pairs))),
        plot_margin_top=0.05
        )
    + p9.xlab("PGC")                                        # ...and set it to the single label you want
)

#fig.save(os.path.join(OUTDIR, f"ligand_gaps_dotplot_{DATASET_TAG}.png"), dpi=150)
fig

## 11. Save

In [ ]:
comp.to_csv(os.path.join(OUTDIR, f"comparison_all_{DATASET_TAG}.csv"), index=False)
gaps.to_csv(os.path.join(OUTDIR, f"actionable_gaps_{DATASET_TAG}.csv"), index=False)
artefacts.to_csv(os.path.join(OUTDIR, f"in_vitro_only_{DATASET_TAG}.csv"), index=False)
vivo.uns["liana_res"].to_csv(os.path.join(OUTDIR, f"liana_invivo_full_{DATASET_TAG}.csv"), index=False)
vitro.uns["liana_res"].to_csv(os.path.join(OUTDIR, f"liana_invitro_full_{DATASET_TAG}.csv"), index=False)

print("wrote to", OUTDIR)
print(f"actionable gaps: {int((comp['class'] == 'actionable').sum())}, "
      f"robust across thresholds: {int(comp['robust'].sum())}")

## 12. Gene Ontology Analysis

Using Enrichr's GO libraries. The background is set to only the genes actually tested in the in vivo LIANA run (via lr["vivo"]), not the whole genome/Enrichr's default, otherwise the enrichment overstates significance for genes that were never even candidates.

In [ ]:
%pip install gseapy   
import gseapy as gp
import textwrap  # add near your other imports at the top

ORGANISM = "human"
FDR_CUTOFF = 0.05

def genes_from_complexes(series):
    """Split each ligand_complex / receptor_complex string on '_' into individual
    gene symbols and return a sorted, de-duplicated list."""
    genes = set()
    for name in series.dropna().astype(str):
        genes.update(name.split("_"))
    return sorted(genes)

def wrap_labels(labels, width=40):
    return [textwrap.fill(str(l), width=width) for l in labels]

# --- gene lists from the actionable class ---
actionable = comp[comp["class"] == "actionable"]
assert len(actionable) > 0, "no rows classed 'actionable' - check the classification step ran"

ligand_genes   = genes_from_complexes(actionable["ligand_complex"])
receptor_genes = genes_from_complexes(actionable["receptor_complex"])
combined_genes = sorted(set(ligand_genes) | set(receptor_genes))

print(f"{len(ligand_genes)} unique ligand genes, "
      f"{len(receptor_genes)} unique receptor genes, "
      f"{len(combined_genes)} combined")

# --- background: every ligand/receptor gene that was testable in vivo,
# not the whole genome. Pulled from lr["vivo"]  ---
vivo_pairs = lr["vivo"].index.to_frame(index=False)
background = sorted(set(genes_from_complexes(vivo_pairs["ligand_complex"])) |
                     set(genes_from_complexes(vivo_pairs["receptor_complex"])))
print(f"{len(background)} genes in background (all vivo-tested ligands/receptors)")

# --- pick the current GO library names rather than hardcoding a year,
# since Enrichr periodically retires old versions (e.g. _2023 -> _2025) ---
available = gp.get_library_name(organism="Human")
def latest(prefix):
    matches = sorted([l for l in available if l.startswith(prefix)], reverse=True)
    return matches[0] if matches else None

GO_GENE_SETS = [g for g in (
    latest("GO_Biological_Process"),
    latest("GO_Molecular_Function"),
    latest("GO_Cellular_Component"),
) if g is not None]
print("using GO libraries:", GO_GENE_SETS)

def run_go(gene_list, label, outdir=OUTDIR):
    if len(gene_list) < 3:
        print(f"skipping {label}: only {len(gene_list)} genes, too few for enrichment")
        return None
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=GO_GENE_SETS,
        organism=ORGANISM,
        background=background,
        outdir=None,   # save manually below instead of letting gseapy write files
    )
    res = enr.results.sort_values("Adjusted P-value")
    res.to_csv(os.path.join(outdir, f"go_enrichment_{label}.csv"), index=False)

    sig = res[res["Adjusted P-value"] < FDR_CUTOFF]
    print(f"{label}: {len(sig)} terms at FDR < {FDR_CUTOFF} (of {len(res)} tested)")

    if len(sig) > 0:
        top = sig.sort_values("Adjusted P-value").head(15).iloc[::-1]
        wrapped_terms = wrap_labels(top["Term"], width=40)
        fig, ax = plt.subplots(figsize=(7, max(3, 0.45 * len(top))))
        ax.barh(wrapped_terms, -np.log10(top["Adjusted P-value"]), color="#2166ac")
        ax.set_xlabel("-log10(adjusted p-value)")
        ax.set_title(f"GO enrichment: {label}")
        plt.tight_layout()
        #plt.savefig(os.path.join(outdir, f"go_enrichment_{label}.png"),
                    #dpi=150, bbox_inches="tight")
        plt.show()
    return res

go_ligand    = run_go(ligand_genes, "actionable_ligands")
go_receptor  = run_go(receptor_genes, "actionable_receptors")
go_combined  = run_go(combined_genes, "actionable_combined")

## KEGG Analysis

Similar structure to above, but using latest Kegg 2021 library instead for comparison

N.B. KEGG pathway names tend to be broader i.e. canonical pathways 

In [ ]:
ORGANISM = "human"
FDR_CUTOFF = 0.05

def genes_from_complexes(series):
    """Split each ligand_complex / receptor_complex string on '_' into individual
    gene symbols and return a sorted, de-duplicated list."""
    genes = set()
    for name in series.dropna().astype(str):
        genes.update(name.split("_"))
    return sorted(genes)

# --- gene lists from the actionable class ---
actionable = comp[comp["class"] == "actionable"]
assert len(actionable) > 0, "no rows classed 'actionable' - check the classification step ran"

ligand_genes   = genes_from_complexes(actionable["ligand_complex"])
receptor_genes = genes_from_complexes(actionable["receptor_complex"])
combined_genes = sorted(set(ligand_genes) | set(receptor_genes))

print(f"{len(ligand_genes)} unique ligand genes, "
      f"{len(receptor_genes)} unique receptor genes, "
      f"{len(combined_genes)} combined")

# --- background: every ligand/receptor gene that was testable in vivo,
# not the whole genome - keeps the enrichment honest about what was actually
# a candidate to begin with. Pulled from lr["vivo"] built earlier. ---
vivo_pairs = lr["vivo"].index.to_frame(index=False)
background = sorted(set(genes_from_complexes(vivo_pairs["ligand_complex"])) |
                     set(genes_from_complexes(vivo_pairs["receptor_complex"])))
print(f"{len(background)} genes in background (all vivo-tested ligands/receptors)")

# --- pick the current KEGG library name rather than hardcoding a year,
# since Enrichr periodically retires old versions ---
available = gp.get_library_name(organism="Human")
def latest(prefix):
    matches = sorted([l for l in available if l.startswith(prefix)], reverse=True)
    return matches[0] if matches else None

KEGG_GENE_SET = latest("KEGG_2021")
assert KEGG_GENE_SET is not None, f"no KEGG library found; available libraries include: {available[:10]}..."
print("using KEGG library:", KEGG_GENE_SET)

def run_kegg(gene_list, label, outdir=OUTDIR):
    if len(gene_list) < 3:
        print(f"skipping {label}: only {len(gene_list)} genes, too few for enrichment")
        return None
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=[KEGG_GENE_SET],
        organism=ORGANISM,
        background=background,
        outdir=None,   # save manually below instead of letting gseapy write files
    )
    res = enr.results.sort_values("Adjusted P-value")
    res.to_csv(os.path.join(outdir, f"kegg_enrichment_{label}.csv"), index=False)

    sig = res[res["Adjusted P-value"] < FDR_CUTOFF]
    print(f"{label}: {len(sig)} pathways at FDR < {FDR_CUTOFF} (of {len(res)} tested)")

    if len(sig) > 0:
        top = sig.sort_values("Adjusted P-value").head(15).iloc[::-1]
        fig, ax = plt.subplots(figsize=(7, max(3, 0.35 * len(top))))
        ax.barh(top["Term"], -np.log10(top["Adjusted P-value"]), color="#2166ac")
        ax.set_xlabel("-log10(adjusted p-value)")
        ax.set_title(f"KEGG enrichment: {label}")
        plt.tight_layout()
        plt.savefig(os.path.join(outdir, f"kegg_enrichment_{label}.png"),
                    dpi=150, bbox_inches="tight")
        plt.show()
    return res

kegg_ligand   = run_kegg(ligand_genes, "actionable_ligands")
kegg_receptor = run_kegg(receptor_genes, "actionable_receptors")
kegg_combined = run_kegg(combined_genes, "actionable_combined")